# 🚀 Tutorial Definitivo de Metaflow (con el dataset Titanic)

¡Bienvenido! En este notebook aprenderás los conceptos fundamentales de **Metaflow**, un framework desarrollado por Netflix para facilitar la construcción y gestión de proyectos de Data Science.

A lo largo de este tutorial, construiremos un pipeline de Machine Learning paso a paso para predecir la supervivencia en el famoso dataset del Titanic.

## 🧠 Conceptos Core de Metaflow que aprenderemos:
1. **Flows y Steps**: La estructura básica (DAG) usando clases y decoradores `@step`.
2. **Estado y Artefactos (`self`)**: Cómo pasar datos entre pasos de forma automática.
3. **Parámetros (`Parameter`)**: Cómo pasar argumentos desde la línea de comandos.
4. **Branching y Joining**: Cómo ejecutar tareas en paralelo (entrenaremos múltiples modelos a la vez) y luego combinar sus resultados.
5. **Client API**: Cómo acceder a los resultados y modelos de ejecuciones pasadas directamente desde este notebook.
6. **Decoradores de robustez**: Menciones a `@retry`, `@catch`, etc.

In [ ]:
# Instalación de dependencias necesarias si no las tienes
!pip install metaflow pandas scikit-learn

## 1. Definiendo el Flujo (The Flow)

En Metaflow, un pipeline se define como una clase de Python que hereda de `FlowSpec`. Cada paso del pipeline es un método decorado con `@step`.
La transición entre pasos se define usando `self.next()`.

Para usar Metaflow desde Jupyter, la práctica más común (y más parecida a la realidad en producción) es escribir el código a un archivo de Python usando el "magic command" `%%writefile` y luego ejecutarlo.

A continuación, crearemos nuestro `TitanicFlow`. Lee atentamente los comentarios en el código, ya que explican cada concepto en detalle.

In [ ]:
%%writefile titanic_flow.py
from metaflow import FlowSpec, step, Parameter, current
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

class TitanicFlow(FlowSpec):
    """
    Un flujo para entrenar modelos predictivos sobre el dataset del Titanic.
    """
    
    # CONCEPT: Parameter
    # Nos permite pasar argumentos al flujo al momento de ejecutarlo.
    test_size = Parameter('test_size', 
                          help='Proporción del dataset para pruebas',
                          default=0.2)

    @step
    def start(self):
        """
        Paso inicial: Cargamos los datos.
        """
        print("Iniciando el flujo. Cargando datos del Titanic...")
        url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
        
        # CONCEPT: Artefactos (State/Artifacts)
        # Al guardar variables en 'self', Metaflow las persiste automáticamente
        # y las hace disponibles en los siguientes pasos (e incluso después de la ejecución).
        self.df = pd.read_csv(url)
        
        # Pasamos al siguiente paso
        self.next(self.preprocess)

    @step
    def preprocess(self):
        """
        Limpieza básica de datos y división en train/test.
        """
        print("Preprocesando datos...")
        # Eliminamos columnas irrelevantes para este ejemplo rápido
        df_clean = self.df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
        
        # Imputamos valores nulos de forma sencilla
        df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
        df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
        
        # Convertimos variables categóricas a numéricas
        df_clean = pd.get_dummies(df_clean, columns=['Sex', 'Embarked'], drop_first=True)
        
        X = df_clean.drop('Survived', axis=1)
        y = df_clean['Survived']
        
        # Usamos el parámetro 'test_size' que definimos al principio
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=42
        )
        
        # CONCEPT: Branching (Bifurcación)
        # En lugar de pasar a un solo paso, pasamos a dos.
        # Metaflow ejecutará 'train_rf' y 'train_lr' en paralelo.
        self.next(self.train_rf, self.train_lr)

    @step
    def train_rf(self):
        """Entrena un Random Forest."""
        print("Entrenando Random Forest...")
        self.model = RandomForestClassifier(random_state=42)
        self.model.fit(self.X_train, self.y_train)
        self.model_name = "Random Forest"
        self.next(self.join)

    @step
    def train_lr(self):
        """Entrena una Regresión Logística."""
        print("Entrenando Regresión Logística...")
        # Aumentamos max_iter para evitar warnings de convergencia
        self.model = LogisticRegression(max_iter=1000, random_state=42)
        self.model.fit(self.X_train, self.y_train)
        self.model_name = "Regresión Logística"
        self.next(self.join)

    # CONCEPT: Joining (Unión)
    # Cuando venimos de múltiples ramas, el paso recibe un argumento extra (inputs),
    # que contiene el estado de cada rama.
    @step
    def join(self, inputs):
        """
        Une las ramas, evalúa los modelos y elige el mejor.
        """
        print("Uniendo las ramas y evaluando modelos...")
        self.results = []
        
        # Iteramos sobre los inputs de cada rama
        for inp in inputs:
            preds = inp.model.predict(inp.X_test)
            acc = accuracy_score(inp.y_test, preds)
            self.results.append({
                'model_name': inp.model_name,
                'accuracy': acc,
                'model': inp.model
            })
            print(f"Modelo {inp.model_name} logró un accuracy de: {acc:.4f}")
        
        # En un join, Metaflow no propaga automáticamente el 'self' de las ramas
        # si hay conflictos (ej: 'self.model' existe en ambas ramas con distinto valor).
        # Por convención, rescatamos lo que necesitemos explícitamente.
        self.X_test = inputs[0].X_test
        self.y_test = inputs[0].y_test
        
        # Seleccionamos el mejor modelo
        self.best_result = max(self.results, key=lambda x: x['accuracy'])
        self.best_model = self.best_result['model']
        
        print(f"🏆 El mejor modelo es {self.best_result['model_name']}!")
        
        self.next(self.end)

    @step
    def end(self):
        """
        Paso final.
        """
        print("¡Flujo completado exitosamente!")
        print(f"Mejor accuracy general: {self.best_result['accuracy']:.4f}")

if __name__ == '__main__':
    TitanicFlow()


## 2. Ejecutando el Flujo

Ahora que hemos definido el flujo en `titanic_flow.py`, podemos ejecutarlo.
La forma clásica de hacerlo es por terminal con el comando `run`.

> **Nota:** Puedes pasar parámetros al ejecutar. Por ejemplo, `--test_size 0.25`. Si no lo pasas, usará el valor por defecto que definimos en la clase (0.2).

In [ ]:
!python titanic_flow.py run --test_size 0.25

## 3. Accediendo a los Datos con la API Cliente (Client API)

¡Una de las características más poderosas de Metaflow! Cada vez que ejecutas un flujo, Metaflow guarda **todo** (datos, modelos, métricas) automáticamente en su almacén de datos local (o en la nube si está configurado, como en AWS S3).

Puedes acceder a ejecuciones pasadas directamente desde tu Jupyter Notebook sin tener que volver a ejecutar nada ni guardar modelos manualmente (ej. con `pickle`).

In [ ]:
from metaflow import Flow, Run, Step

# Obtenemos el flujo por el nombre de la clase
flow = Flow('TitanicFlow')

# Obtenemos la última ejecución exitosa
latest_run = flow.latest_successful_run
print(f"Ejecución ID: {latest_run.id}")

# Podemos acceder directamente a los artefactos (variables 'self') guardados en el pipeline
print("\n--- Resultados de los modelos ---")
for result in latest_run.data.results:
    print(f"{result['model_name']}: {result['accuracy']:.4f}")

print(f"\n🌟 Mejor modelo guardado: {latest_run.data.best_result['model_name']}")

# Incluso podemos extraer el modelo real y hacer predicciones aquí mismo
best_model_recuperado = latest_run.data.best_model
X_test_recuperado = latest_run.data.X_test

# Hacemos una predicción de prueba con el modelo recuperado
sample_pred = best_model_recuperado.predict(X_test_recuperado.iloc[[0]])
resultado_texto = 'Sobrevivió' if sample_pred[0] == 1 else 'No sobrevivió'
print(f"\nPredicción para la primera fila del test set: {resultado_texto}")

## 4. Otros Conceptos Core (Decoradores)

Metaflow brilla en entornos de producción gracias a sus decoradores adicionales. Aunque no los ejecutaremos aquí para no complicar el entorno local, es **crucial** conocerlos:

1. **`@retry`**: Se coloca sobre un `@step`. Si el paso falla (por ejemplo, por un error de red al descargar datos), Metaflow lo reintentará automáticamente.
   ```python
   from metaflow import retry
   
   @retry(times=3)
   @step
   def mi_paso(self): ...
   ```

2. **`@catch`**: Atrapa errores para que el flujo no falle por completo, permitiendo continuar o manejar el error elegantemente.
   ```python
   from metaflow import catch
   
   @catch(var='error_msg')
   @step
   def mi_paso(self): ...
   ```

3. **`@timeout`**: Evita que un paso se quede colgado infinitamente.
   ```python
   from metaflow import timeout
   
   @timeout(minutes=10)
   @step
   def mi_paso(self): ...
   ```

4. **`@conda` / `@pypi` / `@environment`**: (¡Súper importante!) Permite definir dependencias exactas a nivel de paso. Metaflow creará un entorno virtual aislado para ese paso específico en tiempo de ejecución, asegurando reproducibilidad total.
   ```python
   from metaflow import pypi
   
   @pypi(packages={'scikit-learn': '1.0.2'})
   @step
   def train(self): ...
   ```

## Conclusión
Con Metaflow puedes:
- Dividir tu lógica en pasos manejables (**Steps** y **Flows**).
- Olvidarte de guardar archivos intermedios o modelos manualmente (**State/Artifacts** con `self`).
- Entrenar múltiples modelos o procesar datos en paralelo sin hilos complejos (**Branching**).
- Inspeccionar resultados a posteriori de forma interactiva en Notebooks (**Client API**).